# Lakesoul 对象存储向量检索

In [1]:
import os
from pathlib import Path

# 缓存配置必须在 import lakesoul 之前设置：native extension 在 import 时读取环境变量。
CACHE_ROOT = Path(os.environ.get("LAKESOUL_TUTORIAL_CACHE", "/tmp/lakesoul-minio-tutorial"))
os.environ.setdefault("LAKESOUL_VECTOR_CACHE", "1")
os.environ.setdefault("LAKESOUL_VECTOR_MEMORY_BYTES", "512MiB")
os.environ.setdefault("LAKESOUL_CACHE_PATH", str(CACHE_ROOT / "cache"))
os.environ.setdefault("LAKESOUL_CACHE_SIZE", "1GiB")
# 可选：索引文件专用 L2，跨进程重启保留（教程里开启以便观察 index-cache 目录）
os.environ.setdefault("LAKESOUL_INDEX_SSD_BYTES", "256MiB")
os.environ.setdefault("LAKESOUL_VECTOR_LAZY_INDEX", "auto")

from lakesoul import LakeSoulCatalog, LakeSoulSession

# 设置Minio的环境
BUCKET = os.environ.get("MINIO_BUCKET", "lakesoul-tutorial")
ENDPOINT = os.environ.get("MINIO_ENDPOINT", "http://127.0.0.1:9000")
OPTIONS = {
    "fs.s3a.bucket": BUCKET,
    "fs.s3a.endpoint": ENDPOINT,
    "fs.s3a.access.key": os.environ.get("MINIO_ACCESS_KEY", "minioadmin"),
    "fs.s3a.secret.key": os.environ.get("MINIO_SECRET_KEY", "minioadmin"),
    "fs.s3a.path.style.access": "true"  # MinIO 需要 path-style
}


In [2]:
# 建表并写入Minio
import numpy as np
import pyarrow as pa
from lakesoul import LakeSoulCatalog

TABLE = "tutorial_minio_docs"
PREFIX = "tutorial/minio-docx"
PATH = f"s3://{BUCKET}/{PREFIX}"

catalog = LakeSoulCatalog.from_env(object_store_options=OPTIONS)
catalog.drop_table(TABLE, if_exists=True)

schema = pa.schema([
    pa.field("id", pa.int64(), nullable=False),
    pa.field("embedding", pa.list_(pa.field("item", pa.float32()), 8), nullable=False),
])

rows = pa.table({
    "id": pa.array([1, 2, 3, 4, 5, 6], type=pa.int64()),
    "embedding": pa.FixedSizeListArray.from_arrays(
        pa.array(np.array([
            [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.9, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0],
            [0.5, 0.5, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
        ], dtype=np.float32).reshape(-1), type=pa.float32()),
        8
    ),
})

table = catalog.create_table(
    TABLE,
    path=PATH,
    schema=schema,
    primary_keys=["id"],
    hash_bucket_num=2,
    vector_index=[{
        "column": "embedding",
        "dim": 8,
        "nlist": 2,
        "total_bits": 1,
        "metric": "L2",
    }],
)

# write_arrow 提交数据 + 自动构建/提交向量索引
table.write_arrow(rows)

  Reservoir: 3 / 3 vectors (0 MB)


WriteResult(files=(FileInfo(partition='-5', path='s3://lakesoul-tutorial/tutorial/minio-docx/part-zCB6oEsA3AGRMwAf_0000.vortex', size=3056, existing_columns=('id', 'embedding'), row_count=3, other_info=mappingproxy({'physical_format': 'vortex-compact'})), FileInfo(partition='-5', path='s3://lakesoul-tutorial/tutorial/minio-docx/part-zCB6oEsA3AGRMwAf_0001.vortex', size=3192, existing_columns=('id', 'embedding'), row_count=3, other_info=mappingproxy({'physical_format': 'vortex-compact'}))), partitions=mappingproxy({'-5': (FileInfo(partition='-5', path='s3://lakesoul-tutorial/tutorial/minio-docx/part-zCB6oEsA3AGRMwAf_0000.vortex', size=3056, existing_columns=('id', 'embedding'), row_count=3, other_info=mappingproxy({'physical_format': 'vortex-compact'})), FileInfo(partition='-5', path='s3://lakesoul-tutorial/tutorial/minio-docx/part-zCB6oEsA3AGRMwAf_0001.vortex', size=3192, existing_columns=('id', 'embedding'), row_count=3, other_info=mappingproxy({'physical_format': 'vortex-compact'})))}

  Training k-means (2 clusters, 15 iterations)...
  Streaming rotation + quantisation...
  Build complete: 3 vectors, 2 clusters
  ── Phase timing ──
    rotate reservoir:    0.0s
    k-means (15 iter):   0.0s
    stream rotate:       0.0s
    stream GEMM:         0.0s
    stream quantise:     0.0s
    stream append:       0.0s
    flush_pending:       0.0s
  Reservoir: 3 / 3 vectors (0 MB)
  Training k-means (2 clusters, 15 iterations)...
  Streaming rotation + quantisation...
  Build complete: 3 vectors, 2 clusters
  ── Phase timing ──
    rotate reservoir:    0.0s
    k-means (15 iter):   0.0s
    stream rotate:       0.0s
    stream GEMM:         0.0s
    stream quantise:     0.0s
    stream append:       0.0s
    flush_pending:       0.0s


In [3]:
# 从Minio查询
session = catalog.session(
    object_store_options=OPTIONS,
    warehouse_prefix=f"s3://{BUCKET}",
    vector_search={"nprobe": 2, "exact_overfetch": 2},
)

q = [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

# exact：读候选行原始向量做精确重排
exact = table.search(q, top_k=3, rerank="exact", session=session)
print(exact)
# index-only：只按索引距离排序，不读向量列
approx = table.search(q, top_k=3, rerank="index", session=session)
print(approx)
# 带距离 / 过滤 / 投影
detail = table.search(q, top_k=3, with_distance=True, columns=['id'], session=session)
print(detail)
zh = table.search(q, top_k=2, filter="id > 1", session=session)
print(zh)

# 批量
batch = table.search_batch(
    [q, [0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]],
    top_k=2,
    rerank="index",
    nprobe=2,
    session=session,
)
print(batch.ids)

pyarrow.Table
id: int64 not null
----
id: [[1,2,6]]
pyarrow.Table
id: int64 not null
----
id: [[1,2,6]]
pyarrow.Table
id: int64 not null
__lakesoul_vector_distance: double
----
id: [[1,2,6]]
__lakesoul_vector_distance: [[0,0.141421374149721,0.7071067811865476]]
pyarrow.Table
id: int64 not null
----
id: [[2,6]]
[[1, 2], [2, 3]]


[FastScan] Using AVX-512 optimized path


# 缓存使用方法

本节按三种调用方式演示同一套向量缓存：

1. **Python 接口**：`LakeSoulCatalog` / `LakeSoulTable.search`
2. **DataFusion SQL**：`LakeSoulSession.sql`
3. **pgvector 兼容服务**：`postgres-lakesoul` + `psycopg`

缓存只影响速度，不影响查询结果。配置已在第一个 cell 的 `import lakesoul` 之前设置；
如果之前已经 import 过 `lakesoul`，请重启 kernel 后从头运行。

| 变量 | 层 | 本教程值 | 说明 |
|---|---|---|---|
| `LAKESOUL_VECTOR_CACHE` | 总开关 | `1` | `0` 关闭向量 L1/L2 |
| `LAKESOUL_VECTOR_MEMORY_BYTES` | L1 内存 | `512MiB` | 进程内总预算（cluster/snapshot/block） |
| `LAKESOUL_CACHE_PATH` | L2 SSD | `/tmp/lakesoul-minio-tutorial/cache` | 本地缓存根目录 |
| `LAKESOUL_CACHE_SIZE` | L2 SSD | `1GiB` | 通用 `DiskCache` 容量（页/PK index/block） |
| `LAKESOUL_INDEX_SSD_BYTES` | L2 SSD | `256MiB` | 索引文件专用缓存（可选） |
| `LAKESOUL_VECTOR_LAZY_INDEX` | 索引读取 | `auto` | 大索引只加载 probe 的 cluster |

> 这些缓存都写在**本地**。MinIO 上只有数据文件和 `_vector_index/`，没有 payload。

> 对比实验：重启 kernel 后把 `LAKESOUL_CACHE_SIZE` 设为 `0`（只关 L2）或
> `LAKESOUL_VECTOR_CACHE=0`（关向量缓存；当前实现也会关闭通用 `DiskCache`），
> 再运行同样的查询。结果应一致，耗时会变长。


## 1. Python 接口

`LakeSoulTable.search` 直接使用进程级缓存：

- exact 会从 MinIO 读候选行所在的 block，并写入本地 L2；
- 再次 exact 命中 decoded-block cache；
- index-only 只加载索引，不读向量列。

> 缓存开关没有 Python 运行时 API，必须在 `import lakesoul` 之前设置环境变量；
> `session.set_vector_search(...)` / SQL `SET` 只调整检索参数。也没有 Python API 可以直接读取
> 缓存命中率，这里用“查询耗时”和本地缓存文件数观察。
>
> 注意：正常从头运行 notebook 时，前面的“从 Minio 查询”cell 已经跑过一次 exact，
> 所以下面的第 1 次已经是热缓存。要观察真正的冷启动，请重启 kernel，只运行第 1、2 个
> cell（建表）后再运行本节。


In [4]:
import time

# 允许“只运行第 1、2 个 cell（建表）后直接运行本节”：本节自己补齐 q 和 session。
q = globals().get("q", [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])
session = globals().get("session")
if session is None:
    session = catalog.session(
        object_store_options=OPTIONS,
        warehouse_prefix=f"s3://{BUCKET}",
        vector_search={"nprobe": 2, "exact_overfetch": 2},
    )

CACHE_VARS = [
    "LAKESOUL_VECTOR_CACHE",
    "LAKESOUL_VECTOR_MEMORY_BYTES",
    "LAKESOUL_CACHE_PATH",
    "LAKESOUL_CACHE_SIZE",
    "LAKESOUL_INDEX_SSD_BYTES",
    "LAKESOUL_VECTOR_LAZY_INDEX",
]

print("缓存配置（第一个 cell 设置）：")
for key in CACHE_VARS:
    print(f"  {key}={os.environ.get(key)}")

CACHE_DIR = Path(os.environ.get("LAKESOUL_CACHE_PATH", str(CACHE_ROOT / "cache")))
INDEX_CACHE_DIR = Path(
    os.environ.get("LAKESOUL_INDEX_SSD_DIR", str(CACHE_DIR / "index-cache"))
)


def report_local_cache():
    for label, path in (
        ("L2 disk", CACHE_DIR),
        ("index SSD", INDEX_CACHE_DIR),
    ):
        files = [item for item in path.rglob("*") if item.is_file()] if path.is_dir() else []
        total = sum(item.stat().st_size for item in files)
        print(f"  {label:9s}: {len(files):3d} files, {total / 1024 / 1024:8.2f} MiB")


def timed_search(label, **kwargs):
    started = time.perf_counter()
    result = table.search(q, top_k=3, session=session, **kwargs)
    elapsed = (time.perf_counter() - started) * 1000
    print(f"  {label:20s}: {elapsed:8.2f} ms ids={result.column('id').to_pylist()}")
    return result


print("\nPython table.search：")
timed_search("exact 第1次", rerank="exact")
timed_search("exact 第2次", rerank="exact")
timed_search("index-only", rerank="index")

print("\n本地缓存文件：")
if CACHE_DIR.is_dir():
    report_local_cache()
else:
    print("  缓存目录还不存在：确认第一个 cell 的 LAKESOUL_CACHE_PATH 在 import lakesoul 前已设置。")


缓存配置（第一个 cell 设置）：
  LAKESOUL_VECTOR_CACHE=1
  LAKESOUL_VECTOR_MEMORY_BYTES=512MiB
  LAKESOUL_CACHE_PATH=/tmp/lakesoul-minio-tutorial/cache
  LAKESOUL_CACHE_SIZE=1GiB
  LAKESOUL_INDEX_SSD_BYTES=256MiB
  LAKESOUL_VECTOR_LAZY_INDEX=auto

Python table.search：
  exact 第1次           :     2.39 ms ids=[1, 2, 6]
  exact 第2次           :     1.53 ms ids=[1, 2, 6]
  index-only          :     1.24 ms ids=[1, 2, 6]

本地缓存文件：
  L2 disk  :  18 files,     0.02 MiB
  index SSD:  12 files,     0.01 MiB


## 2. DataFusion（SQL）

`LakeSoulSession.sql` 执行的就是 DataFusion 引擎，和上面的 `table.search` 共用同一个进程缓存
（索引、snapshot、decoded block），只是检索参数通过 SQL `SET` 设置。

缓存开关（`LAKESOUL_VECTOR_*` / `LAKESOUL_CACHE_*`）仍然只能通过环境变量控制；
`SET` 只影响 `nprobe` / `rerank` / `exact_overfetch` 等检索参数。

> 因为上一节已经查过，下面的“第1次”通常也是热缓存；第一次 SQL 主要用于确认下推和参数。
> 如果是 Rust 嵌入 DataFusion，缓存配置同样在进程启动前通过环境变量设置；
> SQL、`SET` 和下推行为与本例完全一致，参见 `04-datafusion-embedded.md`。


In [5]:
session.sql("SET lakesoul_vector_search.nprobe = 2")
session.sql("SET lakesoul_vector_search.exact_overfetch = 2")
session.sql("SET lakesoul_vector_search.rerank = 'exact'")

literal = "ARRAY[" + ", ".join(f"{value:.6g}" for value in q) + "]"
sql = f"""
SELECT id, array_distance(embedding, {literal}) AS distance
FROM "lakesoul"."default"."{TABLE}"
ORDER BY array_distance(embedding, {literal})
LIMIT 3
"""


def timed_sql(label, statement):
    started = time.perf_counter()
    result = session.sql(statement)
    elapsed = (time.perf_counter() - started) * 1000
    print(f"  {label:20s}: {elapsed:8.2f} ms ids={result.column('id').to_pylist()}")
    return result


print("DataFusion SQL：")
timed_sql("exact 第1次", sql)
timed_sql("exact 第2次", sql)

session.sql("SET lakesoul_vector_search.rerank = 'index'")
timed_sql("index-only", sql)

plan = session.explain(sql)
print("\n下推检查：LakeSoulVectorSearchExec =", "LakeSoulVectorSearchExec" in plan)
print(plan)


DataFusion SQL：
  exact 第1次           :     7.84 ms ids=[1, 2, 6]
  exact 第2次           :     6.13 ms ids=[1, 2, 6]
  index-only          :     5.81 ms ids=[1, 2, 6]

下推检查：LakeSoulVectorSearchExec = False
+------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
| plan_type                                                  | plan                                                                                                                                                                                                                                                                                                                     

## 3. pgvector 兼容服务

`postgres-lakesoul` 是**独立进程**：缓存配置要加在服务端启动环境里，而不是客户端。
客户端只能通过 SQL 调 `ivfflat.probes` 和 `lakesoul_vector_search.rerank`。

先在另一个终端启动服务（在仓库根目录执行；MinIO 参数按你的环境替换）：

```bash
export LAKESOUL_PG_URL='jdbc:postgresql://127.0.0.1:5432/lakesoul_test?stringtype=unspecified'
export LAKESOUL_PG_USERNAME=lakesoul_test
export LAKESOUL_PG_PASSWORD=lakesoul_test

export BUCKET="${MINIO_BUCKET:-lakesoul-tutorial}"
export ENDPOINT="${MINIO_ENDPOINT:-http://127.0.0.1:9000}"
export ACCESS_KEY="${MINIO_ACCESS_KEY:-minioadmin}"
export SECRET_KEY="${MINIO_SECRET_KEY:-minioadmin}"

export LAKESOUL_VECTOR_CACHE=1
export LAKESOUL_VECTOR_MEMORY_BYTES=512MiB
export LAKESOUL_CACHE_PATH=/tmp/lakesoul-pg-cache
export LAKESOUL_CACHE_SIZE=1GiB
export LAKESOUL_INDEX_SSD_BYTES=256MiB
export LAKESOUL_VECTOR_LAZY_INDEX=auto

./rust/target/release/postgres-lakesoul \
  --bind 127.0.0.1 -p 6543 \
  --warehouse-prefix "s3://$BUCKET" \
  --endpoint "$ENDPOINT" \
  --s3-bucket "$BUCKET" \
  --s3-access-key "$ACCESS_KEY" \
  --s3-secret-key "$SECRET_KEY"
```

服务只读，表由前面的 Python 代码创建；连接 `dbname=default` 可以看到 `tutorial_minio_docs`。
缓存目录（`/tmp/lakesoul-pg-cache`）属于服务进程，客户端只负责查询。

客户端依赖（只需安装一次）：

```bash
python -m pip install "psycopg[binary]" pgvector
# 或者用 uv：
# uv pip install "psycopg[binary]" pgvector
```

> `pgvector` 这里是 Python 客户端包（提供 `Vector` 和 `register_vector`）；
> LakeSoul 的 `postgres-lakesoul` 已经实现了 pgvector 兼容协议，不需要在 PostgreSQL
> 里安装 pgvector 扩展。


In [6]:
DSN = os.environ.get(
    "LAKESOUL_PG_SERVICE_DSN",
    "host=127.0.0.1 port=6543 user=lakesoul_test dbname=default",
)

missing = []
try:
    import psycopg
except ImportError:
    missing.append("psycopg[binary]")
try:
    import pgvector  # noqa: F401
except ImportError:
    missing.append("pgvector")

if missing:
    print("跳过 pgvector 示例：缺少依赖", ", ".join(missing))
    print("安装命令：python -m pip install 'psycopg[binary]' pgvector")
    print("或者：uv pip install 'psycopg[binary]' pgvector")
else:
    import numpy as np
    from pgvector.psycopg import register_vector

    try:
        conn = psycopg.connect(DSN, autocommit=True, connect_timeout=3)
    except Exception as error:  # noqa: BLE001 - tutorial diagnostics
        print("pgvector 兼容服务未启动，跳过。启动方法见上一节。")
        print(error)
    else:
        register_vector(conn)
        conn.execute("SET ivfflat.probes = 2")
        conn.execute("SET lakesoul_vector_search.rerank = 'exact'")
        query = np.asarray(q, dtype=np.float32)
        statement = """
            SELECT id, embedding <-> %s AS distance
            FROM tutorial_minio_docs
            ORDER BY embedding <-> %s
            LIMIT 3
        """

        def timed_pg(label):
            started = time.perf_counter()
            rows = conn.execute(statement, (query, query)).fetchall()
            elapsed = (time.perf_counter() - started) * 1000
            print(f"  {label:20s}: {elapsed:8.2f} ms rows={rows}")
            return rows

        print("pgvector 服务端缓存：")
        timed_pg("exact 第1次")
        timed_pg("exact 第2次")

        conn.execute("SET lakesoul_vector_search.rerank = 'index'")
        timed_pg("index-only")

        plan = conn.execute("EXPLAIN " + statement, (query, query)).fetchall()
        print(
            "\n下推检查：LakeSoulVectorSearchExec =",
            any("LakeSoulVectorSearchExec" in str(row) for row in plan),
        )
        conn.close()


pgvector 服务端缓存：
  exact 第1次           :    39.15 ms rows=[(1, 0.0), (2, 0.141421374149721), (6, 0.7071067811865476)]
  exact 第2次           :     3.09 ms rows=[(1, 0.0), (2, 0.141421374149721), (6, 0.7071067811865476)]
  index-only          :     2.90 ms rows=[(1, 0.0), (2, 0.141421374149721), (6, 0.7071067811865476)]

下推检查：LakeSoulVectorSearchExec = True


## 清理（可选）

教程结束后可以删除表、S3 前缀和本地缓存：

```python
# catalog.drop_table(TABLE, if_exists=True)
# import shutil
# shutil.rmtree(CACHE_ROOT, ignore_errors=True)
```

pgvector 兼容服务的缓存目录（如 `/tmp/lakesoul-pg-cache`）属于服务进程，需要单独清理。


In [ ]:
catalog.drop_table(TABLE, if_exists=True)
import shutil
shutil.rmtree(CACHE_ROOT, ignore_errors=True)

: 